# Introduction to jbubble

We start by importing the necessary modules, including JAX.

In [ ]:
import time
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
from jbubble import (
    Units,
    SaveSpec,
    Bubble,
    Pulse,
    run_simulation,
)
import jbubble.shapes as shapes

# Enable 64-bit precision for better stability in physical simulations
from jax import config
config.update("jax_enable_x64", True)

## 1. Running a Single Simulation

We'll start by defining the simulation parameters: units, a bubble, and an acoustic pulse.

In [ ]:
# Create a default bubble with initial radius R0 = 3 microns
# Currently we just have one bubble definition, but later on we can easily add others
bubble = Bubble(R0=3.0e-6)
print(bubble)

In [ ]:
# Create a driving pulse
pulse = Pulse(
    freq=800e3, # [Hz]
    pressure=100e3, # [Pa]
    shape=shapes.Triangle(), # Can also try: "Sine", "Square", "Sawtooth", etc.
    cycle_num=10, # Number of cycles in the pulse
    initial_time=1e-6, # Offset time [s]
    apply_hann=False, # Whether to apply a Hann window to the pulse
)

# Visualise the pulse
time_axis = jnp.linspace(0, 20e-6, 1000)
plt.figure(figsize=(10, 4))
plt.plot(time_axis, pulse(time_axis), color='orange')
plt.xlabel("Time (s)")
plt.ylabel("Pressure (Pa)")
plt.grid()

In [ ]:
# Run the simulation

# Define how we want to save the results (number of time points)
save_spec = SaveSpec(num_samples=1000)

# We want to avoid numerical instabilities due to dealing with small (e.g. micrometres) and large (megahertz) numbers
# So we define a "units" object, which scales the SI units of quantities appropriately
# Here we use the default scaling: 1e-6 for length, 1e-6 for time, and 1e-15 for mass
units = Units()

# Finally, we can run the simulation
print("Running simulation...")
start_time = time.time()
result = run_simulation(
    bubble=bubble,
    pulse=pulse,
    units=units,
    save_spec=save_spec,
)
end_time = time.time()
print(f"Simulation took {end_time - start_time:.4f} seconds")

In [ ]:
time_axis = result.ts # [s]
pressure_input = pulse(time_axis) # [Pa]
radius_output = result.radius # [m]

# Plot the results with shared time axis
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8), sharex=True)

# Top plot: Driving Pressure
ax1.plot(time_axis * 1e6, pressure_input / 1e3, color="orange")
ax1.set_xlabel("Time (µs)")
ax1.set_ylabel("Driving Pressure (kPa)")
ax1.grid(True)

# Bottom plot: Radius
ax2.plot(time_axis * 1e6, radius_output * 1e6, color='blue')
ax2.set_ylabel("Radius (µm)")
ax2.set_title("Bubble Radius vs Time")
ax2.grid(True)

plt.tight_layout()
plt.show()

## 2. JIT Compilation Speedup

JAX allows us to Just-In-Time (JIT) compile functions to XLA, which can significantly speed up execution. We'll compare the time it takes to run the simulation with and without the compilation overhead.

In [ ]:
# As a quick demonstration, consider the simple function
def f(x):
    return jnp.sin(x)

# JIT compile it
jit_f = jax.jit(f)

# Test input
x_test = jnp.linspace(0, 10, 1000)

# First run (includes compilation)
print("First run (with compilation)...")
start = time.perf_counter()
result1 = jit_f(x_test)
_ = result1.block_until_ready() # Ensure computation is actually finished
end = time.perf_counter()
time_first = end - start
print(f"   Time: {time_first:.6f} seconds")

# Second run (execution only)
print("Second run (jitted, no compilation)...")
start = time.perf_counter()
result2 = jit_f(x_test)
_ = result2.block_until_ready()
end = time.perf_counter()
time_second = end - start
print(f"   Time: {time_second:.6f} seconds")

print(f"\nSpeedup factor: {time_first / time_second:.1f}x")

In [ ]:
# Now we do the same for the bubble simulation

# JIT compile the simulation function
jit_run_simulation = jax.jit(run_simulation)

print("1. First run (includes compilation)...")
start1 = time.perf_counter()
# We run the jitted function. The first time, it compiles.
res1 = jit_run_simulation(
    bubble=bubble,
    pulse=pulse,
    units=units,
    save_spec=save_spec,
)
# Block until ready to ensure we measure the full computation time
_ = res1.ys.block_until_ready()
end1 = time.perf_counter()
print(f"   Time: {end1 - start1:.4f} seconds")

print("\n2. Second run (execution only)...")
start2 = time.perf_counter()
res2 = jit_run_simulation(
    bubble=bubble,
    pulse=pulse,
    units=units,
    save_spec=save_spec,
)
_ = res2.ys.block_until_ready()
end2 = time.perf_counter()
print(f"   Time: {end2 - start2:.4f} seconds")

print(f"\nSpeedup factor (vs first run): {(end1 - start1) / (end2 - start2):.1f}x")

## 3. Parameter Sweep: Frequency vs Initial Radius

We will now generate a colormap plot showing the **Maximum Expansion Ratio** ($R_{max} / R_0$) as a function of:
- Excitation Frequency (X-axis)
- Initial Radius (Y-axis)

We'll use `jax.vmap` to vectorize the simulation over a grid of parameters. This allows us to write our code for the single-input case, and then easily run over many batches.

In [ ]:
# Define a function that runs a single simulation given frequency and initial radius
from jbubble.bubble import Bubble
from jbubble.pulse import Pulse
import jbubble.shapes as shapes

units = Units()
save_spec = SaveSpec(num_samples=1000)

def run_single_param(r0, freq):
    # Create bubble with the given initial radius
    bubble_param = Bubble(
        R0=r0,
        R_buckle=0.99 * r0,
        gamma=1.07,
        chi=0.38,
        mu_L=0.00089,
        kappa_s=2.4e-9,
        rho_L=1000.0,
        c_L=1498.0,
        P_amb=101.3e3,
        sigma_L=72e-3,
    )


    # Create pulse with the given frequency
    pulse_param = Pulse(
        freq=freq,
        pressure=100e3,
        shape=shapes.Sine(),
        cycle_num=5,
        initial_time=1e-6,
        apply_hann=False,
    )

    # Run simulation
    result = run_simulation(
        bubble=bubble_param,
        pulse=pulse_param,
        units=units,
        save_spec=save_spec,
        window_s=20e-6,
    )

    return result

# Vectorize over both radius and frequency
vectorized_run = jax.vmap(run_single_param)

# Run the parameter sweep
print("Running parameter sweep...")
start_time = time.time()

r0_values = jnp.linspace(1.0e-6, 5.0e-6, 100)    # Initial radius range [m]
freq_values = jnp.linspace(0.5e6, 1.5e6, 100)   # Frequency range [Hz]
r0_grid, freq_grid = jnp.meshgrid(r0_values, freq_values)
r0_flat = r0_grid.ravel()
freq_flat = freq_grid.ravel()

results_flat = vectorized_run(r0_flat, freq_flat)
_ = results_flat.radius.block_until_ready()
end_time = time.time()

tot_time = end_time - start_time
per_sim_time = tot_time / freq_grid.size
print(f"Parameter sweep took {tot_time:.2f} s ({per_sim_time:.2e} s/sim)")
print(f"All converged? = {results_flat.converged.all()}")

# Reshape back to 2D grid
results_grid = jax.tree.map(lambda x: x.reshape(*r0_grid.shape, *x.shape[1:]), results_flat)
results_grid

### Open question here...
Why is there a weird hard junction of instability?

In [ ]:
expansion_ratios = (results_grid.radius.max(axis=-1) / results_grid.bubble.R0)
# plot the heatmap of expansion ratios
plt.figure(figsize=(10, 8))
plt.pcolormesh(r0_grid * 1e6, freq_grid / 1e6, expansion_ratios, shading='auto', cmap='viridis')
plt.colorbar(label='Max Expansion Ratio ($R_{max}/R_0$)')
plt.xlabel('Initial Radius ($\mu$m)')
plt.ylabel('Frequency (MHz)')
plt.title('Bubble Expansion Ratio Heatmap')
plt.gca().invert_yaxis()
plt.show()

For example, notice here the abrupt onset of instability?

In [ ]:
# Empirical stability cliff at around mid-point of r0_values
stability_cliff = r0_values.shape[0] // 2
target_point_stable = [5, stability_cliff]
target_point_unstable = [5, stability_cliff + 1]

plt.figure(figsize=(15, 8))
plt.plot(results_grid.radius[*target_point_stable], color='k', label=f"radius = {r0_values[stability_cliff]:.2e}") # Stable
plt.plot(results_grid.radius[*target_point_unstable], color='r', label=f"radius = {r0_values[stability_cliff + 1]:.2e}") # Onset of instability in radius?
plt.legend()